# Linguistic complexity by gender on Simpsons dialogue

This notebook reproduces `lingcomplexity.py` (week3) but reads the raw Simpsons data files directly instead of using `convokit`.

It expects the following files to already be available locally in the Colab runtime, under `/content/sample_data/` (the standard Colab `sample_data` folder, visible in the Files pane):
- `simpsons_speakers.txt` (columns: `speaker_id`, `gender`)
- `simpsons_lines.txt` (columns: `line_id`, `speaker_id`, `text`)

Drag these two files directly into the `sample_data` folder in the Colab Files pane before running the cells below.

We then:
1. Split utterances into texts spoken by male vs. female characters.
2. Group utterances into 10-utterance chunks ("sentgrams") per gender.
3. Use spaCy + TRUNAJOD to compute linguistic complexity/diversity metrics per chunk.
4. Compare the distributions for male vs. female with boxplots.

In [ ]:
!pip install -q TRUNAJOD
!python -m spacy download en_core_web_sm -q

In [ ]:
import pandas as pd
import seaborn as sns
import spacy
import TRUNAJOD.ttr
from TRUNAJOD import surface_proxies

import nltk
from nltk.corpus import stopwords
from tqdm import tqdm
import matplotlib.pyplot as plt

nltk.download("stopwords")

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["figure.dpi"] = 200

## 1. Load the data

We read the speakers file to build a `speaker_id -> gender` lookup, then read the lines file and split the utterance texts into `male` and `female` lists based on the speaker's gender. This replaces the `convokit.Corpus` used in the original script.

In [ ]:
SPEAKERS_PATH = "/content/sample_data/simpsons_speakers.txt"
LINES_PATH = "/content/sample_data/simpsons_lines.txt"


def load_speaker_genders(path):
    genders = {}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 2:
                continue
            speaker_id, gender = fields[0], fields[1]
            genders[speaker_id] = gender
    return genders


def process_data():
    print("Loading data...")
    genders = load_speaker_genders(SPEAKERS_PATH)

    male = []
    female = []
    with open(LINES_PATH, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 3:
                continue
            _line_id, speaker_id, text = fields[0], fields[1], fields[2]
            gender = genders.get(speaker_id, "?")
            if gender == "f":
                female.append(text)
            elif gender == "m":
                male.append(text)
    print("Done.")
    print("No. male:", len(male), "No. female:", len(female))

    # stopword removal is computed but unused below, kept for parity with the original script
    stopwords_set = set(stopwords.words("english"))
    male_clean = []
    female_clean = []
    for text in male:
        text = " ".join([tok for tok in text.split(" ") if tok not in stopwords_set])
        male_clean.append(text)
    for text in female:
        text = " ".join([tok for tok in text.split(" ") if tok not in stopwords_set])
        female_clean.append(text)

    # return male_clean[:1000], female_clean[:1000]  # Check stopword filtering effect
    return male[:1000], female[:1000]


male, female = process_data()

## 2. Measure linguistic complexity

Utterances are grouped into overlapping 10-utterance chunks per gender. For each chunk we compute lexical diversity (MTLD), lexical density, POS dissimilarity and connection-words ratio, then plot the distributions per gender.

**Homework** (from the original script): see the [TRUNAJOD.ttr docs](https://trunajod20.readthedocs.io/en/latest/api_reference/ttr.html) for other metrics to add. E.g. `TRUNAJOD.ttr.yule_k` can be added to `features` as `"yule_k": []`, computed for male/female below, and plotted alongside the existing boxplots.

In [ ]:
def measure_complexity(male, female):
    nlp = spacy.load("en_core_web_sm", disable=["ner", "textcat"])

    features = {
        "lexical_diversity_mltd": [],
        "lexical_density": [],
        "pos_dissimilarity": [],
        "connection_words_ratio": [],
        "gender": [],
    }

    nsent = 10
    male_sentgrams = [" ".join(male[i:i + nsent]) for i in range(len(male) - nsent + 1)]
    female_sentgrams = [" ".join(female[i:i + nsent]) for i in range(len(female) - nsent + 1)]

    print("Male...")
    for text in tqdm(male_sentgrams):
        doc = nlp(text)
        features["lexical_diversity_mltd"].append(TRUNAJOD.ttr.lexical_diversity_mtld(doc))
        features["lexical_density"].append(surface_proxies.lexical_density(doc))
        features["pos_dissimilarity"].append(surface_proxies.pos_dissimilarity(doc))
        features["connection_words_ratio"].append(surface_proxies.connection_words_ratio(doc))
        features["gender"].append("male")

    print("Female...")
    for text in tqdm(female_sentgrams):
        doc = nlp(text)
        features["lexical_diversity_mltd"].append(TRUNAJOD.ttr.lexical_diversity_mtld(doc))
        features["lexical_density"].append(surface_proxies.lexical_density(doc))
        features["pos_dissimilarity"].append(surface_proxies.pos_dissimilarity(doc))
        features["connection_words_ratio"].append(surface_proxies.connection_words_ratio(doc))
        features["gender"].append("female")

    df = pd.DataFrame(features)
    _fig, axes = plt.subplots(2, 2)
    sns.boxplot(x="gender", y="lexical_diversity_mltd", data=df, ax=axes[0, 0])
    sns.boxplot(x="gender", y="lexical_density", data=df, ax=axes[0, 1])
    sns.boxplot(x="gender", y="pos_dissimilarity", data=df, ax=axes[1, 0])
    sns.boxplot(x="gender", y="connection_words_ratio", data=df, ax=axes[1, 1])
    plt.savefig("linguistic_complexity_diversity.png")
    plt.show()
    return df

In [ ]:
df = measure_complexity(male, female)
print("Done. See linguistic_complexity_diversity.png")